In [1]:
import numpy as np
import time
from dataclasses import dataclass
from typing import Callable, Optional
import matplotlib.pyplot as plt

In [2]:
from simulator import generate_hierarchical
from posteriors import *
from samplers_hierarchical import *

In [3]:
rng = np.random.default_rng(221)

print("Generating data")
ds = generate_hierarchical(rng=rng, truth_model = "power_law")
K = len(ds.seasons)
print(f"K = {K}, "
     f"True means = {ds.phi_k}")


Generating data
K = 10, True means = [1.23580097e-05 9.27441331e-06 8.02758220e-06 8.77680051e-06
 1.34651853e-05 1.00865273e-05 1.08705475e-05 1.33043578e-05
 6.35077821e-06 1.12390328e-05]


In [4]:
param_names = []

for i in range(K):
    param_names.append(f"log_phi_{i+1}")

param_names.extend(["gamma", "log_eta", "delta", "mu_phi", "log_sigma_phi"])
param_names

['log_phi_1',
 'log_phi_2',
 'log_phi_3',
 'log_phi_4',
 'log_phi_5',
 'log_phi_6',
 'log_phi_7',
 'log_phi_8',
 'log_phi_9',
 'log_phi_10',
 'gamma',
 'log_eta',
 'delta',
 'mu_phi',
 'log_sigma_phi']

In [5]:

true_values = {
    **{f"log_phi_{k+1}": np.log(ds.phi_k[k]) for k in range(K)},
    "gamma": ds.true_params["gamma"],
    "log_eta": np.log(ds.true_params["eta"]),
    "delta": ds.true_params["delta"],
    "mu_phi": ds.true_params["mu_phi"],
    "log_sigma_phi": np.log(ds.true_params["sigma_phi"]),
}


In [6]:
mu_phi_true = ds.true_params["mu_phi"]
sigma_phi_true = ds.true_params["sigma_phi"]

z_k_true = (np.log(ds.phi_k) - mu_phi_true) / sigma_phi_true

theta_true = np.concatenate([
    z_k_true,
    [
        ds.true_params["gamma"],
        np.log(ds.true_params["eta"]),
        ds.true_params["delta"],
        mu_phi_true,
        np.log(sigma_phi_true),
    ],
])

theta_init = theta_true + rng.normal(0, 0.05, size=K+5)

In [7]:
prior_type = "lognormal_gamma"
parameterization = "noncentered"

def log_post(theta):
    return log_posterior_hierarchical(theta, ds.seasons, parameterization, prior_type)

def grad_log_post(theta):
    return grad_log_posterior_hierarchical(theta, ds.seasons, parameterization, prior_type)

In [8]:
rwmh_results = run_multiple_chains(
        run_rwmh,
        theta_init=theta_init,
        n_chains=4,
        init_strategy="jitter",
        init_scale=0.001,
        rng=rng,
        log_posterior_fn=log_post,
        n_iterations=100000,
        n_burnin=20000,
        adapt_until=20000,
        adapt_proposal=True,
        param_names=param_names,
    )

Iteration 5000/100000: accept rate = 0.249, scale = 0.626, elapsed = 1.5s
Iteration 10000/100000: accept rate = 0.239, scale = 0.361, elapsed = 2.9s
Iteration 15000/100000: accept rate = 0.239, scale = 0.663, elapsed = 4.3s
Iteration 20000/100000: accept rate = 0.240, scale = 0.563, elapsed = 5.8s
Iteration 25000/100000: accept rate = 0.233, scale = 0.563, elapsed = 7.4s
Iteration 30000/100000: accept rate = 0.234, scale = 0.563, elapsed = 8.8s
Iteration 35000/100000: accept rate = 0.233, scale = 0.563, elapsed = 10.3s
Iteration 40000/100000: accept rate = 0.236, scale = 0.563, elapsed = 11.8s
Iteration 45000/100000: accept rate = 0.234, scale = 0.563, elapsed = 13.3s
Iteration 50000/100000: accept rate = 0.232, scale = 0.563, elapsed = 14.7s
Iteration 55000/100000: accept rate = 0.230, scale = 0.563, elapsed = 16.3s
Iteration 60000/100000: accept rate = 0.229, scale = 0.563, elapsed = 17.7s
Iteration 65000/100000: accept rate = 0.226, scale = 0.563, elapsed = 19.0s
Iteration 70000/100

In [9]:
rwmh_cov = estimate_dense_precond_from_rwmh(rwmh_results, ridge=1e-6)

In [10]:
mala_results = run_multiple_chains(
        run_mala,
        theta_init=theta_init,
        n_chains=4,
        init_strategy="jitter",
        init_scale=0.001,
        rng=rng,
        log_posterior_fn=log_post,
        grad_log_posterior_fn=grad_log_post,
        n_iterations=100000,
        n_burnin=20000,
        step_size=1e-3,
        adapt_step=True,
        adapt_until=20000,
        target_accept=0.65,
        param_names=param_names,
        precond=rwmh_cov,
        adapt_precond=False,
        precond_type="dense",
        normalize_precond=True,
    )

Iteration 5000/100000: accept rate = 0.827, step_size = 0.0136311, elapsed = 2.7s
Iteration 10000/100000: accept rate = 0.716, step_size = 0.01416, elapsed = 5.6s
Iteration 15000/100000: accept rate = 0.675, step_size = 0.0117676, elapsed = 8.4s
Iteration 20000/100000: accept rate = 0.670, step_size = 0.0122242, elapsed = 11.2s
Iteration 25000/100000: accept rate = 0.658, step_size = 0.0122242, elapsed = 13.9s
Iteration 30000/100000: accept rate = 0.657, step_size = 0.0122242, elapsed = 16.7s
Iteration 35000/100000: accept rate = 0.674, step_size = 0.0122242, elapsed = 19.3s
Iteration 40000/100000: accept rate = 0.680, step_size = 0.0122242, elapsed = 22.0s
Iteration 45000/100000: accept rate = 0.685, step_size = 0.0122242, elapsed = 24.6s
Iteration 50000/100000: accept rate = 0.692, step_size = 0.0122242, elapsed = 27.3s
Iteration 55000/100000: accept rate = 0.691, step_size = 0.0122242, elapsed = 30.0s
Iteration 60000/100000: accept rate = 0.673, step_size = 0.0122242, elapsed = 32.7

In [11]:
# print_diagnostics_multi({"RWMH": rwmh_results,"MALA": mala_results}, true_values = true_values)

In [12]:
print_diagnostics_multi_hierarchical(
    {"RWMH": rwmh_results, "MALA": mala_results},
    parameterization="noncentered",
    K=10,
    latent_display="raw",
    true_values=true_values,
)

save_traceplots_multi_hierarchical(
    mala_results,
    "traceplots_mala_noncentered_logphi.png",
    parameterization="noncentered",
    K=10,
    latent_display="log_phi",
)


save_traceplots_multi_hierarchical(
    rwmh_results,
    "traceplots_rwmh_noncentered_logphi.png",
    parameterization="noncentered",
    K=10,
    latent_display="log_phi",
)


Sampler      Chains  Accept%  Time(s)ESS(z_1)ESS(z_2)ESS(z_3)ESS(z_4)ESS(z_5)ESS(z_6)ESS(z_7)ESS(z_8)ESS(z_9)ESS(z_10)ESS(gamma)ESS(log_eta)ESS(delta)ESS(mu_phi)ESS(log_sigma_phi)
------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
RWMH              4    0.229    112.1       2020       2129       1898       2122       1633       2160       2848       2239       2163       2101       1209        606       1226        610        861
MALA              4    0.653    212.8       2366       2374       2025       1876       1222       1977       2519       2037       2055       2637        994        530       1434       1104        773

Sampler      Chains  Accept%  Time(s)Rhat(z_1)Rhat(z_2)Rhat(z_3)Rhat(z_4)Rhat(z_5)Rhat(z_6)Rhat(z_7)Rhat(z_8)Rhat(z_9)Rhat(z_10)Rhat(gamma)Rhat(log_eta)Rhat(delta)Rhat(mu_phi)Rhat(log_sigma_phi)


In [13]:
log_posterior_hierarchical(theta_true, ds.seasons, "centered")

np.float64(-27771899.857627146)

In [14]:
theta_mala = np.array([-0.1009,0.3353,0.0046,-0.0057,0.3186,-0.1673,-0.1265,
                      -0.1843,-0.1314,-0.1431,2.4964,6.2411,3.6875,1.5748,-1.4372])
log_posterior_hierarchical(theta_mala, ds.seasons, "centered")

np.float64(-90240661309.22896)